# API-Football: PL Team Schedule Collector

Fetches full fixture lists (all competitions) for every Premier League team across multiple seasons.

**Request budget:** 1 (team list) + 20 teams × 5 seasons = **101 requests** — just over the 100/day free limit, so the script skips cached files and only fetches what's missing.

Run cells top to bottom. Cached responses are saved to `infra/data/json/api_football/fixtures_raw/` so you can re-run freely without burning quota.

## 1. Setup

In [6]:
import json
import time
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv
import os

# Load API key from .env at project root
project_root = Path("../../../../").resolve()
load_dotenv(project_root / ".env")

API_KEY = os.environ.get("API_FOOTBALL_KEY", "")
assert API_KEY, "API_FOOTBALL_KEY not found — check your .env file"
print(f"Key loaded: {API_KEY[:6]}...")

Key loaded: b5fabf...


In [7]:
BASE_URL     = "https://v3.football.api-sports.io"
PL_LEAGUE_ID = 39
SEASONS      = [2022, 2023, 2024]  # Free plan: 2022–2024 only

OUTPUT_DIR = project_root / "infra" / "data" / "json" / "api_football" / "fixtures_raw"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FLAT_CSV = OUTPUT_DIR.parent / "fixtures_flat.csv"

REQUEST_DELAY = 7  # Free plan: 10 req/min max → 6s minimum, use 7 to be safe

print(f"Output directory: {OUTPUT_DIR}")

Output directory: /Users/admin/dev/algobetting/infra/data/json/api_football/fixtures_raw


## 2. API helpers

In [8]:
def api_get(endpoint, params, retries=3):
    for attempt in range(retries):
        resp = requests.get(
            f"{BASE_URL}/{endpoint}",
            headers={"x-apisports-key": API_KEY},
            params=params,
            timeout=30,
        )
        if resp.status_code == 429:
            wait = 65  # back off a full minute
            print(f"  429 rate limit hit — waiting {wait}s before retry {attempt+1}/{retries}...")
            time.sleep(wait)
            continue
        resp.raise_for_status()
        remaining = resp.headers.get("x-ratelimit-requests-remaining", "?")
        print(f"  quota remaining today: {remaining}")
        data = resp.json()
        if data.get("errors"):
            raise RuntimeError(f"API error: {data['errors']}")
        return data
    raise RuntimeError("Exceeded retries due to rate limiting.")

# ── Debug: confirm auth ─────────────────────────────────────────────────────
resp = requests.get(f"{BASE_URL}/status", headers={"x-apisports-key": API_KEY}, timeout=30)
print("HTTP status:", resp.status_code)
print(json.dumps(resp.json(), indent=2))

HTTP status: 200
{
  "get": "status",
  "parameters": [],
  "errors": [],
  "results": 0,
  "paging": {
    "current": 1,
    "total": 1
  },
  "response": {
    "account": {
      "firstname": "Louis",
      "lastname": "Porter",
      "email": "louis.porter26@gmail.com"
    },
    "subscription": {
      "plan": "Free",
      "end": "2026-04-16T00:30:03+00:00",
      "active": true
    },
    "requests": {
      "current": 9,
      "limit_day": 100
    }
  }
}


## 3. Fetch PL team list

In [9]:
teams_cache = OUTPUT_DIR.parent / "pl_teams_2024.json"

if teams_cache.exists():
    teams = json.loads(teams_cache.read_text())
    print(f"Loaded {len(teams)} teams from cache.")
else:
    print("Fetching PL team list...")
    data = api_get("teams", {"league": PL_LEAGUE_ID, "season": 2024})
    teams = [{"id": t["team"]["id"], "name": t["team"]["name"]} for t in data["response"]]
    if not teams:
        raise RuntimeError(f"Got 0 teams — full response was: {data}")
    teams_cache.write_text(json.dumps(teams, indent=2))
    print(f"Found {len(teams)} teams.")

for t in teams:
    print(f"  {t['id']:>6}  {t['name']}")

Loaded 20 teams from cache.
      33  Manchester United
      34  Newcastle
      35  Bournemouth
      36  Fulham
      39  Wolves
      40  Liverpool
      41  Southampton
      42  Arsenal
      45  Everton
      46  Leicester
      47  Tottenham
      48  West Ham
      49  Chelsea
      50  Manchester City
      51  Brighton
      52  Crystal Palace
      55  Brentford
      57  Ipswich
      65  Nottingham Forest
      66  Aston Villa


## 4. Fetch fixtures — one request per team × season

Already-cached files are skipped, so this is safe to re-run.

In [10]:
total     = len(teams) * len(SEASONS)
done      = 0
fetched   = 0
cache_hit = 0

for team in teams:
    for season in SEASONS:
        done += 1
        cache_path = OUTPUT_DIR / f"{team['id']}_{season}.json"

        if cache_path.exists():
            cache_hit += 1
            print(f"[{done}/{total}] CACHED  {team['name']} {season}")
            continue

        print(f"[{done}/{total}] FETCH   {team['name']} {season}")
        time.sleep(REQUEST_DELAY)
        data = api_get("fixtures", {"team": team["id"], "season": season})
        fixtures = data.get("response", [])
        cache_path.write_text(json.dumps(fixtures, indent=2))
        print(f"          → {len(fixtures)} fixtures saved")
        fetched += 1

print(f"\nDone. {fetched} fetched, {cache_hit} from cache.")

[1/60] CACHED  Manchester United 2022
[2/60] CACHED  Manchester United 2023
[3/60] CACHED  Manchester United 2024
[4/60] CACHED  Newcastle 2022
[5/60] CACHED  Newcastle 2023
[6/60] CACHED  Newcastle 2024
[7/60] CACHED  Bournemouth 2022
[8/60] CACHED  Bournemouth 2023
[9/60] FETCH   Bournemouth 2024
  quota remaining today: 89
          → 50 fixtures saved
[10/60] FETCH   Fulham 2022
  quota remaining today: 88
          → 50 fixtures saved
[11/60] FETCH   Fulham 2023
  quota remaining today: 87
          → 50 fixtures saved
[12/60] FETCH   Fulham 2024
  quota remaining today: 86
          → 47 fixtures saved
[13/60] FETCH   Wolves 2022
  quota remaining today: 85
          → 57 fixtures saved
[14/60] FETCH   Wolves 2023
  quota remaining today: 84
          → 53 fixtures saved
[15/60] FETCH   Wolves 2024
  quota remaining today: 83
          → 48 fixtures saved
[16/60] FETCH   Liverpool 2022
  quota remaining today: 82
          → 59 fixtures saved
[17/60] FETCH   Liverpool 2023
  quot

## 5. Flatten to DataFrame & save CSV

In [11]:
all_fixtures = []
for path in OUTPUT_DIR.glob("*.json"):
    all_fixtures.extend(json.loads(path.read_text()))

# Deduplicate — each fixture appears in both home and away team's response
seen = set()
unique = []
for f in all_fixtures:
    fid = f["fixture"]["id"]
    if fid not in seen:
        seen.add(fid)
        unique.append(f)

print(f"Total raw: {len(all_fixtures)}, unique: {len(unique)}")

if not unique:
    print("No fixtures found yet — run the fetch cell first.")
else:
    rows = []
    for f in unique:
        rows.append({
            "fixture_id":     f["fixture"]["id"],
            "date":           f["fixture"]["date"],
            "status":         f["fixture"]["status"]["short"],
            "venue":          f["fixture"]["venue"]["name"],
            "league_id":      f["league"]["id"],
            "league_name":    f["league"]["name"],
            "league_country": f["league"]["country"],
            "season":         f["league"]["season"],
            "round":          f["league"]["round"],
            "home_team_id":   f["teams"]["home"]["id"],
            "home_team":      f["teams"]["home"]["name"],
            "away_team_id":   f["teams"]["away"]["id"],
            "away_team":      f["teams"]["away"]["name"],
            "home_goals":     f["goals"]["home"],
            "away_goals":     f["goals"]["away"],
            "ht_home":        f["score"]["halftime"]["home"],
            "ht_away":        f["score"]["halftime"]["away"],
        })

    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["date"], utc=True).dt.tz_convert("Europe/London")
    df = df.sort_values("date").reset_index(drop=True)

    df.to_csv(FLAT_CSV, index=False)
    print(f"Saved → {FLAT_CSV}")
    df.head()

Total raw: 3326, unique: 2181
Saved → /Users/admin/dev/algobetting/infra/data/json/api_football/fixtures_flat.csv


## 6. Quick sanity check — fixture counts by competition

In [12]:
df.groupby(["league_name", "season"]).size().unstack(fill_value=0)

season,2022,2023,2024
league_name,,,
Championship,0,135,0
Community Shield,1,1,1
EFL Trophy,4,0,0
Emirates Cup,1,1,1
FA Cup,42,51,45
FIFA Club World Cup,0,2,0
Friendlies Clubs,148,96,96
League Cup,42,37,42
League One,46,0,0
